In [ ]:
def reparer_total_charges(df):
    """
    Convertit TotalCharges en numérique et traite les trous révélés.
    Renvoie le dataframe réparé.
    """
    df = df.copy() 

    valeurs_uniques = df["TotalCharges"].unique()
    test_conversion = pd.to_numeric(df["TotalCharges"], errors="coerce")
    pct_nan_apres = test_conversion.isna().mean()

    if pct_nan_apres > 0.5:
        print("⛔ REFUS : plus de 50% de NaN après conversion, colonne probablement pas numérique.")
        print(f"   ({pct_nan_apres:.1%} de NaN détectés)")
        return df


    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    nb_trous = df["TotalCharges"].isna().sum()
    print(f"🔍 Trous démasqués dans TotalCharges : {nb_trous}")

    if nb_trous > 0:
      
        print("\n🔎 Détail des lignes concernées :")
        print(df[df["TotalCharges"].isna()][["customerID", "tenure", "MonthlyCharges", "TotalCharges"]].to_string())

   
        mediane = df["TotalCharges"].median()
        df["TotalCharges"] = df["TotalCharges"].fillna(mediane)
        print(f"\n✅ Imputation par la médiane : {mediane:.2f}")
        print(f"   Trous restants : {df['TotalCharges'].isna().sum()}")

    print(f"\n📊 Nouveau type de TotalCharges : {df['TotalCharges'].dtype}")
    return df


df = reparer_total_charges(df)
print("\n✅ Phase 2 terminée.")

In [ ]:
print("Type TotalCharges :", df["TotalCharges"].dtype)
print("NaN restants :", df["TotalCharges"].isna().sum())
print("Aperçu :")
df[["tenure", "MonthlyCharges", "TotalCharges"]].describe()

In [ ]:


def verifier_format_numerique(df, colonne):
    """
    Vérifie si une colonne texte contient des virgules comme séparateur décimal.
    Alerte si c'est le cas.
    """
    if df[colonne].dtype == object:
        masque_virgule = df[colonne].astype(str).str.contains(",", na=False)
        nb_virgules = masque_virgule.sum()
        if nb_virgules > 0:
            print(f"⚠️  {nb_virgules} valeurs avec virgule détectées dans '{colonne}'.")
            print("   → Remplacer les virgules par des points avant conversion.")
            print("   Exemple :", df.loc[masque_virgule, colonne].iloc[0])
        else:
            print(f"✅ Pas de virgule détectée dans '{colonne}'.")
    else:
        print(f"ℹ️  '{colonne}' n'est pas de type texte, pas de vérification nécessaire.")

df_original = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
verifier_format_numerique(df_original, "TotalCharges")

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Forme du dataset :", df.shape)
print("\nTypes des colonnes :")
print(df.dtypes)
print("\nAperçu des 5 premières lignes :")
df.head()

In [ ]:
def audit_qualite(df):
    """
    Affiche un rapport de santé du dataset.
    Montre : dimensions, types, % manquants par colonne,
    et la répartition de la cible Churn.
    """
    print("=" * 55)
    print("         RAPPORT D'AUDIT QUALITÉ")
    print("=" * 55)

    # 1. Dimensions
    nb_lignes, nb_colonnes = df.shape
    print(f"\n📐 Dimensions : {nb_lignes} lignes × {nb_colonnes} colonnes")

    # 2. Types de colonnes
    print("\n📋 Types des colonnes :")
    print(df.dtypes.to_string())

    # 3. Valeurs manquantes
    nb_manquants = df.isna().sum()
    pct_manquants = (df.isna().mean() * 100).round(2)
    tableau_manquants = pd.DataFrame({
        "manquants": nb_manquants,
        "pourcent": pct_manquants
    })
    colonnes_avec_trous = tableau_manquants[tableau_manquants["manquants"] > 0]

    print("\n🕳️  Valeurs manquantes :")
    if colonnes_avec_trous.empty:
        print("   Aucun NaN détecté (attention : des trous peuvent être cachés, voir Phase 2)")
    else:
        print(colonnes_avec_trous.sort_values("pourcent", ascending=False).to_string())

    # 4. Doublons
    nb_doublons = df.duplicated().sum()
    print(f"\n♻️  Doublons : {nb_doublons}")

    # 5. Équilibre de la cible Churn
    if "Churn" in df.columns:
        print("\n🎯 Répartition de la cible Churn :")
        counts = df["Churn"].value_counts()
        pcts = df["Churn"].value_counts(normalize=True) * 100
        for val in counts.index:
            print(f"   {val} : {counts[val]} ({pcts[val]:.1f}%)")
        # Alerte si déséquilibre
        ratio_min = pcts.min()
        if ratio_min < 30:
            print(f"   ⚠️  ATTENTION : cible déséquilibrée ({ratio_min:.1f}% pour la classe minoritaire)")
            print("      → L'accuracy seule sera trompeuse demain. Regarder aussi recall/F1.")
    else:
        print("\n⚠️  Colonne 'Churn' introuvable dans ce dataset.")

    print("\n" + "=" * 55)


# On appelle la fonction sur le dataset complet
audit_qualite(df)

In [ ]:
def encoder_features(df):
    """
    Encode toutes les colonnes catégorielles.
    Renvoie un dataframe 100% numérique, prêt pour un modèle.
    """
    df = df.copy()

    if "customerID" in df.columns:
        df = df.drop(columns=["customerID"])
        print("🗑️  customerID supprimé (identifiant, pas une feature)")

    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
    print("✅ Churn encodé : Yes=1, No=0")

    colonnes_oui_non = [
        "Partner", "Dependents", "PhoneService",
        "PaperlessBilling", "MultipleLines",
        "OnlineSecurity", "OnlineBackup", "DeviceProtection",
        "TechSupport", "StreamingTV", "StreamingMovies"
    ]
   
    for col in colonnes_oui_non:
        if col in df.columns:
            df[col] = (df[col] == "Yes").astype(int)
    print(f"✅ {len(colonnes_oui_non)} colonnes Yes/No encodées en 0/1")

    if "gender" in df.columns:
        df["gender"] = df["gender"].map({"Male": 1, "Female": 0})
        print("✅ gender encodé : Male=1, Female=0")

    if "Contract" in df.columns:
        ordre_contrat = {"Month-to-month": 0, "One year": 1, "Two year": 2}
        df["Contract"] = df["Contract"].map(ordre_contrat)
        print("✅ Contract encodé en ordinal (0=mensuel, 1=annuel, 2=deux ans)")

    
    colonnes_one_hot = [col for col in df.select_dtypes(include="object").columns
                        if col != "Churn"] 

    if colonnes_one_hot:
        print(f"\n🔄 One-Hot Encoding sur : {colonnes_one_hot}")
        df = pd.get_dummies(df, columns=colonnes_one_hot, drop_first=False)
        # On convertit les bool en int pour avoir des 0/1 propres
        for col in df.select_dtypes(include="bool").columns:
            df[col] = df[col].astype(int)

    print(f"\n📐 Dimensions après encodage : {df.shape}")
    print(f"   (Avant : 21 colonnes → Après : {df.shape[1]} colonnes)")

    reste_object = df.select_dtypes(include="object").columns.tolist()
    if reste_object:
        print(f"\n⚠️  Colonnes encore en texte : {reste_object}")
    else:
        print("✅ Aucune colonne texte restante, dataset 100% numérique.")

    return df


df_encode = encoder_features(df)
print("\n--- Aperçu des premières colonnes ---")
df_encode.head(3)

In [ ]:

df_test_id = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
nb_clients_uniques = df_test_id["customerID"].nunique()
print(f"⚠️  Si on One-Hot encodait customerID :")
print(f"   {nb_clients_uniques} valeurs uniques → {nb_clients_uniques} nouvelles colonnes !")
print("   C'est l'explosion de dimensions. customerID doit toujours être supprimé.")

In [ ]:
def detecter_outliers_iqr(df, colonne):
  
    if colonne not in df.columns:
        print(f"⛔ Colonne '{colonne}' introuvable.")
        return None, None, None

    valeurs = df[colonne].dropna()

    Q1 = valeurs.quantile(0.25)
    Q3 = valeurs.quantile(0.75)
    IQR = Q3 - Q1

    borne_basse = Q1 - 1.5 * IQR
    borne_haute = Q3 + 1.5 * IQR

    outliers = df[(df[colonne] < borne_basse) | (df[colonne] > borne_haute)]
    nombre_outliers = len(outliers)

    return borne_basse, borne_haute, nombre_outliers


colonnes_num = ["tenure", "MonthlyCharges", "TotalCharges"]

print("=" * 55)
print("       ANALYSE DES OUTLIERS (règle IQR)")
print("=" * 55)

for col in colonnes_num:
    bb, bh, nb = detecter_outliers_iqr(df, col)
    print(f"\n📊 {col}")
    print(f"   Bornes normales : [{bb:.1f} ; {bh:.1f}]")
    print(f"   Outliers détectés : {nb}")

    if nb > 0:
        outliers_df = df[(df[col] < bb) | (df[col] > bh)]
        nb_churn_dans_outliers = outliers_df["Churn"].value_counts().get("Yes", 0)
        pct_churn = nb_churn_dans_outliers / nb * 100 if nb > 0 else 0
        print(f"   Dont churners : {nb_churn_dans_outliers} ({pct_churn:.1f}%)")
        print("   → Décision : GARDER (cas réels, pas des erreurs de saisie)")
    else:
        print("   → Pas d'outliers sur cette colonne.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Boxplots des colonnes numériques", fontsize=14, fontweight="bold")

for i, col in enumerate(colonnes_num):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(col)
    axes[i].set_ylabel("Valeur")

plt.tight_layout()
plt.savefig("boxplots_outliers.png", dpi=100, bbox_inches="tight")
plt.show()
print("Graphique sauvegardé : boxplots_outliers.png")

In [ ]:
print("Sensibilité du seuil IQR sur MonthlyCharges :")
for facteur in [1.0, 1.5, 3.0]:
    valeurs = df["MonthlyCharges"].dropna()
    Q1 = valeurs.quantile(0.25)
    Q3 = valeurs.quantile(0.75)
    IQR = Q3 - Q1
    bb = Q1 - facteur * IQR
    bh = Q3 + facteur * IQR
    nb = ((df["MonthlyCharges"] < bb) | (df["MonthlyCharges"] > bh)).sum()
    print(f"   Facteur {facteur} → {nb} outliers détectés")
print("\n→ 'Outlier' n'est pas une vérité absolue, c'est un seuil que VOUS choisissez.")

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def rapport_multicolinearite(df_encoded, colonnes_num):
   
    cols_disponibles = [c for c in colonnes_num if c in df_encoded.columns]

    matrice_corr = df_encoded[cols_disponibles].corr()
    plt.figure(figsize=(8, 6))
    sns.heatmap(matrice_corr, annot=True, fmt=".2f", cmap="coolwarm",
                center=0, linewidths=0.5)
    plt.title("Corrélations entre colonnes numériques")
    plt.tight_layout()
    plt.savefig("heatmap_correlations.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Heatmap sauvegardée : heatmap_correlations.png")

    X = df_encoded[cols_disponibles].dropna()

    vif_valeurs = []
    for i in range(X.shape[1]):
        try:
            val = variance_inflation_factor(X.values, i)
        except Exception:
            val = float("inf")  
        vif_valeurs.append(val)

    vif_df = pd.DataFrame({
        "variable": cols_disponibles,
        "VIF": [round(v, 1) for v in vif_valeurs]
    })

    print("\n📊 VIF par variable :")
    print(vif_df.to_string(index=False))

    problematiques = vif_df[vif_df["VIF"] > 5]
    if not problematiques.empty:
        print("\n⚠️  Variables au VIF > 5 (multicolinéarité problématique) :")
        for _, row in problematiques.iterrows():
            print(f"   {row['variable']} : VIF = {row['VIF']}")
    else:
        print("\n Aucune multicolinéarité détectée (tous VIF ≤ 5).")

    return vif_df



vif_avant = rapport_multicolinearite(df_encode, ["tenure", "MonthlyCharges", "TotalCharges"])

In [ ]:


df_encode = df_encode.drop(columns=["TotalCharges"])
print("🗑️  TotalCharges supprimée.")

print("\nVIF après suppression :")
vif_apres = rapport_multicolinearite(df_encode, ["tenure", "MonthlyCharges"])

In [ ]:
df_test_dup = df_encode.copy()
df_test_dup["tenure_copie"] = df_test_dup["tenure"] 

print("Test colonnes dupliquées :")
vif_test = rapport_multicolinearite(df_test_dup, ["tenure", "MonthlyCharges", "tenure_copie"])
print("\n→ Le VIF de tenure_copie doit être infini (ou très grand) : c'est normal.")
del df_test_dup 

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def features_discriminantes(df_encoded, cible="Churn"):

    if cible not in df_encoded.columns:
        print(f"⛔ Colonne cible '{cible}' introuvable.")
        return

    X = df_encoded.drop(columns=[cible])
    y = df_encoded[cible]

    corr_cible = df_encoded.corr()[cible].drop(cible).abs().sort_values(ascending=False)

    
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X, y)
    importances_rf = pd.Series(rf.feature_importances_, index=X.columns)
    importances_rf = importances_rf.sort_values(ascending=False)

    top_n = 10
    comparaison = pd.DataFrame({
        "Rang_Corrélation": range(1, top_n + 1),
        "Feature_Corr": corr_cible.head(top_n).index,
        "Score_Corr": corr_cible.head(top_n).values.round(3),
        "Rang_RF": range(1, top_n + 1),
        "Feature_RF": importances_rf.head(top_n).index,
        "Score_RF": importances_rf.head(top_n).values.round(3),
    })

    print("=" * 75)
    print("     TOP 10 FEATURES — Corrélation VS Random Forest")
    print("=" * 75)
    print(comparaison.to_string(index=False))

    plt.figure(figsize=(10, 6))
    importances_rf.head(15).sort_values().plot(kind="barh", color="steelblue")
    plt.title("Top 15 features (importance Random Forest)")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.savefig("importance_features.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Graphique sauvegardé : importance_features.png")

    return corr_cible, importances_rf


corr_result, rf_result = features_discriminantes(df_encode)